In [1]:
import warnings

from matplotlib import pyplot as plt

warnings.filterwarnings("ignore")
import torch

import pandas as pd
import numpy as np
import scanpy as sc
import os
import yaml
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['R_HOME'] = r"E:\R-4.5.1"
os.environ['R_USER'] = 'E:\Anaconda\envs\Spacross\Lib\site-packages\rpy2'
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.decomposition import PCA



# %%
import sys
sys.path.append(r"E:\pycharm\project4")
import MACWT as TOOLS

def load_data(config):
    adata = sc.read(r'E:\pycharm\project1\datasets\osmFISH\adata.h5ad')
    adata.var_names_make_unique()

    ##### Load layer_guess label, if have
    adata.obs['layer_guess'] = adata.obs['cluster']

    edge_index = TOOLS.graph_construction(adata, config['data']['k_cutoff'])

    adata.layers['count'] = adata.X
    sc.pp.filter_genes(adata, min_cells=50)
    sc.pp.filter_genes(adata, min_counts=10)
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer='count', n_top_genes=config['data']['top_genes'])
    adata = adata[:, adata.var['highly_variable'] == True]
    sc.pp.scale(adata)

    adata.obsm['X_pca'] = adata.X
    return adata, edge_index


In [ ]:
with open('E:\pycharm\project1\Config\osmFISH.yaml', 'r', encoding='utf-8') as f:
    config = yaml.load(f.read(), Loader=yaml.FullLoader)
    # ... 你贴的这些代码 ...
adata, edge_index = load_data(config)

# 保存为 S3RL 需要的文件
adata.write('E:/pycharm/project3/datasets/osmFISH/processed.h5ad')

In [ ]:
adata, edge_index = load_data(config)

In [ ]:
num_clusters = 11
# %%
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = TOOLS.SC_pipeline(adata, edge_index=edge_index, num_clusters=num_clusters, device=device, config=config,
                 imputation=False)
# %%
net.trian()
# %%
enc_rep, recon = net.process()
adata.obsm['latent'] = enc_rep
adata.obsm['recon'] = recon

In [ ]:
# %%
clusType = "mclust"
adata.obs[clusType] = TOOLS.clustering(z=enc_rep, n_clust=num_clusters, num_seed=1, method=clusType)
adata.obs[clusType] = TOOLS.refine_label(adata, 50, key=clusType)

In [ ]:
ARI, ACC, DIS = TOOLS.get_metrics(adata, 'layer_guess', clusType)
print(f"ARI: {round(ARI, 4)} ACC: {round(ACC, 4)} ")

In [ ]:
sc.pl.spatial(adata, color=[clusType],
              title='ARI:' + str(round(ARI, 3))+ ' ACC:' + str(round(ACC, 3)), show=True, spot_size=500, save=False)